# Antiderivative Operator with DeepONet

Learn the antiderivative operator

$$G(u)(y) = \int_0^y u(s)\, ds, \quad y \in [0, 1]$$

with a Deep Operator Network ([Lu et al., 2021](https://arxiv.org/abs/1910.03193)). The branch net
encodes $u$ from its values at fixed sensor points, the trunk net encodes the query location $y$, and
their dot product gives $G(u)(y)$.

In [ ]:
# Install keras-sciml, then choose a backend: "tensorflow", "torch" or "jax"
!pip install keras-sciml
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Hyperparameters

`M_SENSORS` is the number of points $u$ is sampled at, `N_POINTS` the number of query locations $y$.

In [ ]:
N_SAMPLES = 64
M_SENSORS = 30
N_POINTS = 40
LATENT_DIM = 32
HIDDEN_UNITS = 64
EPOCHS = 40
BATCH_SIZE = 16
LEARNING_RATE = 0.01

## Generate the data

Input functions are random Fourier series
$u(x) = \sum_{k=1}^{4} a_k \sin(k \pi x) + b_k \cos(k \pi x)$ with decaying coefficients, so their
antiderivatives are known in closed form.

In [ ]:
import keras
import numpy as np

from keras_sciml.losses import RelativeL2Loss
from keras_sciml.models import DeepONet

keras.utils.set_random_seed(42)


def generate_antiderivative_data(n_samples, m_sensors, n_points, seed=42):
    rng = np.random.default_rng(seed)
    x_sensors = np.linspace(0.0, 1.0, m_sensors, dtype=np.float32)
    y_coords = np.linspace(0.0, 1.0, n_points, dtype=np.float32)[:, None]

    k = np.arange(1, 5, dtype=np.float32)
    scales = 1.0 / (k ** 1.5)
    a = rng.standard_normal((n_samples, len(k))).astype(np.float32) * scales
    b = rng.standard_normal((n_samples, len(k))).astype(np.float32) * scales

    # u at the sensor locations: (n_samples, m_sensors)
    kx = np.pi * k[:, None] * x_sensors[None, :]
    u_branch = a @ np.sin(kx) + b @ np.cos(kx)

    # int_0^y sin(k pi s) ds = (1 - cos(k pi y)) / (k pi),  int_0^y cos(k pi s) ds = sin(k pi y) / (k pi)
    ky = np.pi * k[:, None] * y_coords.T
    int_sin = (1.0 - np.cos(ky)) / (np.pi * k[:, None])
    int_cos = np.sin(ky) / (np.pi * k[:, None])

    g_target = a @ int_sin + b @ int_cos  # (n_samples, n_points)
    y_batch = np.tile(y_coords[None, :, :], (n_samples, 1, 1))  # (n_samples, n_points, 1)
    return u_branch, y_batch, g_target


u_branch, y_batch, g_target = generate_antiderivative_data(N_SAMPLES, M_SENSORS, N_POINTS)

n_train = int(0.8 * N_SAMPLES)
u_train, y_train, g_train = u_branch[:n_train], y_batch[:n_train], g_target[:n_train]
u_test, y_test, g_test = u_branch[n_train:], y_batch[n_train:], g_target[n_train:]
print(f"Branch input: {u_branch.shape}, trunk input: {y_batch.shape}, target: {g_target.shape}")

## Define the model

In `"cartesian"` mode every input function is evaluated at every query location, so each sample is
one function together with all `N_POINTS` query locations.

In [ ]:
model = DeepONet(
    latent_dim=LATENT_DIM,
    hidden_layers=3,
    hidden_units=HIDDEN_UNITS,
    activation="tanh",
    mode="cartesian",
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=RelativeL2Loss(),
    metrics=[RelativeL2Loss(name="rel_l2")],
)

## Train

In [ ]:
history = model.fit(
    [u_train, y_train],
    g_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=([u_test, y_test], g_test),
    verbose=2,
)

## Evaluate

In [ ]:
test_loss = float(model.evaluate([u_test, y_test], g_test, verbose=0)[0])
print(f"Test relative L2 error: {test_loss:.4f}")